# MAD Stage 3 — Unsloth GRPO Training
### Model: Qwen2.5-7B-Instruct (full bfloat16, A100 40GB) · LoRA r=16

**Training pipeline:**
1. **SFT warm-up** (skipped by default — GRPO's parse-failure penalty teaches format)
2. **GRPO fine-tuning** (2 epochs on 2,484 records) — optimizes calibrated accuracy via Brier reward

**Inputs:** Upload `mad_v2_grpo.jsonl` and `mad_v2_sft.jsonl` from Stage 2.  
**Outputs:** LoRA adapter weights + training logs.

**Data summary:**
- GRPO: 2,484 rows | 414 claims × 3 agents × 2 rounds | avg Brier = 0.865 | 97.4% positive
- SFT: 1,221 rows | best R1 completions per claim×agent | avg Brier = 0.932 | all reward > 0.5

**Reward function:** Symmetric Brier — `R = 1 - 2*(p - v_label)²`  
Parse failure penalty: **−1.0** (forces the model to produce valid JSON).

**Why NOT 4-bit?** `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` causes a fatal dtype mismatch:  
bitsandbytes NF4 dequantization produces `float32` tensors during GRPO's inference phase,  
but Unsloth's `matmul_lora` kernel expects `bfloat16` → `RuntimeError: Half and Float`.  
A100 40GB has plenty of VRAM for the full 7B bfloat16 model (~14 GB), so 4-bit is unnecessary.

In [ ]:
# CELL 1 — Install Unsloth + dependencies
# Colab A100 install (CUDA 12.x)
%%capture
import sys

# Unsloth latest — includes UnslothGRPOTrainer (auto-patches trl on import)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

# Confirm versions
import subprocess
for pkg in ['unsloth', 'trl', 'peft', 'transformers']:
    v = subprocess.run([sys.executable, '-m', 'pip', 'show', pkg],
                       capture_output=True, text=True).stdout
    for line in v.splitlines():
        if line.startswith('Version'):
            print(f'{pkg}: {line}')
            break
print('Install complete')

In [ ]:
# CELL 2 — GPU check + imports
import subprocess, json, re, os, random
import torch
from datasets import Dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig, GRPOConfig, GRPOTrainer

gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
).stdout.strip()
print(f'GPU: {gpu}')
print(f'CUDA: {torch.version.cuda}')
print(f'bfloat16 supported: {is_bfloat16_supported()}')
print(f'Torch: {torch.__version__}')

In [ ]:
# CELL 3 — Upload JSONL files from Stage 2
from google.colab import files

print('Upload your Stage 2 output files:')
print('  Required: mad_v2_grpo.jsonl  (2484 rows)')
print('  Required: mad_v2_sft.jsonl   (1221 rows)')
print()

uploaded = files.upload()

# Locate uploaded files
GRPO_PATH = None
SFT_PATH  = None

for fname in uploaded:
    dest = f'/content/{os.path.basename(fname)}'
    if not os.path.exists(dest):
        os.rename(fname, dest)
    if 'grpo' in fname.lower():
        GRPO_PATH = dest
    elif 'sft' in fname.lower():
        SFT_PATH  = dest

if not GRPO_PATH:
    # Try default name
    for f in os.listdir('/content'):
        if 'grpo' in f.lower() and f.endswith('.jsonl'):
            GRPO_PATH = f'/content/{f}'
        elif 'sft' in f.lower() and f.endswith('.jsonl'):
            SFT_PATH  = f'/content/{f}'

# Load and validate
grpo_data = [json.loads(l) for l in open(GRPO_PATH)]
sft_data  = [json.loads(l) for l in open(SFT_PATH)]

# Filter out IDK claims (v_label is None) from GRPO
grpo_data_filtered = [r for r in grpo_data if r.get('v_label') is not None]

print(f'\nGRPO file: {GRPO_PATH}')
print(f'  Total rows:          {len(grpo_data)}')
print(f'  After IDK filter:    {len(grpo_data_filtered)}')
rewards = [r['brier_reward'] for r in grpo_data_filtered if r.get('brier_reward') is not None]
print(f'  Avg brier_reward:    {sum(rewards)/len(rewards):.4f}')
print(f'  Positive rewards:    {sum(1 for r in rewards if r > 0)} ({sum(1 for r in rewards if r > 0)*100//len(rewards)}%)')
verdicts = {}
for r in grpo_data_filtered:
    v = r['verdict']
    verdicts[v] = verdicts.get(v, 0) + 1
print(f'  Verdict dist:        {verdicts}')
print()
print(f'SFT file:  {SFT_PATH}')
print(f'  Total rows:          {len(sft_data)}')
sft_r = [r['metadata']['brier'] for r in sft_data if r.get('metadata',{}).get('brier')]
if sft_r:
    print(f'  Avg brier_reward:    {sum(sft_r)/len(sft_r):.4f}')
print('\n✓ Files loaded')

In [ ]:
# CELL 4 — Config
# ── Model ────────────────────────────────────────────────────────────────────
# NON-4bit model: A100 40GB has 40GB VRAM. 7B bfloat16 = ~14GB. Plenty of room.
# 4-bit (bnb-4bit) causes a bitsandbytes + Unsloth GRPO kernel dtype conflict
# where 4-bit dequant produces float32 but GRPO expects bfloat16 — unresolvable.
# Use the full bfloat16 model to completely avoid the 4-bit code path.
MODEL_NAME    = 'unsloth/Qwen2.5-7B-Instruct'   # full bfloat16, NO 4-bit
LOAD_IN_4BIT  = False                            # False — use full precision
MAX_SEQ_LEN   = 1536

# ── LoRA ─────────────────────────────────────────────────────────────────────
LORA_R        = 16
LORA_ALPHA    = 16
LORA_DROPOUT  = 0
LORA_TARGETS  = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                  'gate_proj', 'up_proj', 'down_proj']

# ── SFT warm-up (optional — skipped by default, see Cell 7) ──────────────────
SFT_LR        = 2e-4
SFT_EPOCHS    = 1
SFT_BATCH     = 2
SFT_GRAD_ACC  = 4
SFT_WARMUP    = 10

# ── GRPO ─────────────────────────────────────────────────────────────────────
GRPO_LR           = 5e-6
GRPO_EPOCHS       = 2
GRPO_BATCH        = 1
GRPO_GRAD_ACC     = 4
GRPO_NUM_GEN      = 4      # 4 completions per prompt — minimum for GRPO variance
GRPO_MAX_PROMPT   = 1024
GRPO_MAX_COMP     = 512
GRPO_MAX_GRAD     = 0.1
GRPO_WARMUP_STEPS = 50

# ── Output ───────────────────────────────────────────────────────────────────
SFT_OUTPUT_DIR  = '/content/sft_output'
GRPO_OUTPUT_DIR = '/content/grpo_output'
ADAPTER_DIR     = '/content/mad_grpo_adapter'

print('✓ Config set')
print(f'  Model:          {MODEL_NAME}')
print(f'  4-bit quant:    {LOAD_IN_4BIT}  ← False avoids bnb/GRPO dtype conflict')
print(f'  LoRA:           r={LORA_R}, alpha={LORA_ALPHA}')
print(f'  GRPO:           {GRPO_EPOCHS} epochs, lr={GRPO_LR}, gens={GRPO_NUM_GEN}')
print()
print('VRAM estimate on A100 40GB:')
print('  7B bfloat16 base model : ~14.0 GB')
print('  LoRA adapter (r=16)    :  ~0.3 GB')
print('  Optimizer (adamw_8bit) :  ~1.5 GB')
print('  GRPO activations       :  ~6.0 GB (4 gens × seq_len)')
print('  Total estimated        : ~22 GB  ← well within 40GB')


In [ ]:
# CELL 5 — Load model + LoRA
#
# KEY FIXES vs. earlier attempts:
#   1. load_in_4bit=False  → eliminates bitsandbytes NF4 dequant (the first dtype bug)
#   2. use_gradient_checkpointing=False  → eliminates Unsloth's custom GC (the second dtype bug)
#
# WHY gradient_checkpointing=False fixes the crash:
#   Unsloth's 'unsloth' GC pre-allocates activation buffers in float16 to save VRAM.
#   During backward recomputation X arrives as float32 (from the bfloat16 forward path).
#   matmul_lora then sees: out=Half, B.to(X.dtype)=Float → RuntimeError: Half and Float.
#   A100 40GB has ~18 GB headroom after the full 7B model — GC is not needed.

import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from transformers import GenerationConfig

# Auto-select dtype: bfloat16 on A100/H100, float16 on T4/V100
DTYPE = torch.bfloat16 if is_bfloat16_supported() else torch.float16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name        = MODEL_NAME,
    max_seq_length    = MAX_SEQ_LEN,
    load_in_4bit      = LOAD_IN_4BIT,   # False — no bitsandbytes, no NF4 dequant
    dtype             = DTYPE,
    trust_remote_code = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    target_modules             = LORA_TARGETS,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = 'none',
    use_gradient_checkpointing = False,   # ← CRITICAL: disables Unsloth's custom GC
    random_state               = 3407,
    use_rslora                 = False,
    loftq_config               = None,
)

# Guarantee all trainable (LoRA) params are in the correct dtype
# Belt-and-suspenders: even if Unsloth initializes them as float32, cast them now
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(DTYPE)

# Override model's baked-in generation_config
# GRPO internally calls model.generate(); clearing temperature prevents conflict warnings
model.generation_config = GenerationConfig(
    temperature  = None,
    do_sample    = False,
    pad_token_id = tokenizer.eos_token_id,
    eos_token_id = tokenizer.eos_token_id,
)

# ── Summary ──────────────────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
lora_dtypes      = set(p.dtype for p in model.parameters() if p.requires_grad)

print(f'✓ Model loaded: {MODEL_NAME}')
print(f'  Precision:           {DTYPE}  ({"bfloat16 — A100 native" if is_bfloat16_supported() else "float16 — T4 fallback"})')
print(f'  4-bit quant:         {LOAD_IN_4BIT}  ← False, no bitsandbytes')
print(f'  Gradient checkpt:    False  ← no Unsloth GC, no float16 buffer pre-alloc')
print(f'  Total params:        {total_params/1e6:.1f}M')
print(f'  Trainable (LoRA):    {trainable_params/1e6:.2f}M ({100*trainable_params/total_params:.2f}%)')
print(f'  LoRA dtype:          {lora_dtypes}  ← must be {{{DTYPE}}}')
print(f'  GPU memory used:     {torch.cuda.memory_allocated()/1024**3:.2f} GB')
print(f'  GPU memory total:    {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
print()
vram_free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3
print(f'  VRAM remaining:      {vram_free:.1f} GB  ← GRPO needs ~8 GB for 4 generations')
if vram_free < 10:
    print('  ⚠ Low VRAM — reduce GRPO_NUM_GEN to 2 in Cell 4 if training OOMs')

In [ ]:
# CELL 6 — Prepare SFT dataset
# SFT format: system + user + assistant (full conversation)
# Only R1 completions with Brier > 0.5 (1221 records)

def format_sft_sample(row):
    """Format as Qwen2.5 chat template: system + user + assistant."""
    messages = [
        {'role': 'system',    'content': row['system']},
        {'role': 'user',      'content': row['prompt']},
        {'role': 'assistant', 'content': row['completion']},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

sft_examples = [{'text': format_sft_sample(r)} for r in sft_data]
sft_dataset  = Dataset.from_list(sft_examples)

# Sanity check: token length distribution
sample_tokens = tokenizer(sft_examples[0]['text'], return_tensors='pt')
n_tokens = sample_tokens['input_ids'].shape[1]
print(f'Sample #0 token length: {n_tokens}')
print(f'SFT dataset size: {len(sft_dataset)}')
print()
print('Sample (first 200 chars):')
print(sft_examples[0]['text'][:200])
print('...')
print('\n✓ SFT dataset ready')

In [ ]:
# CELL 7 — SFT warm-up training (OPTIONAL — READ BEFORE RUNNING)
# ──────────────────────────────────────────────────────────────────────────────
# ⚠️  IMPORTANT: Running SFT and GRPO in the same Colab session causes a dtype
#    conflict. bitsandbytes 4-bit NF4 always dequantizes to float16, but SFT
#    promotes LoRA weights to bfloat16. matmul_lora then crashes with:
#    "RuntimeError: self and mat2 must have the same dtype, got Half and Float"
#
# RECOMMENDED: SKIP THIS CELL. Go directly to Cell 8 → Cell 9 → Cell 10.
#    The GRPO reward function teaches format through parse failure penalty (-1.0).
#    After 50-100 steps the model learns to produce valid JSON naturally.
#
# ONLY run this cell if:
#   - You plan to DOWNLOAD the SFT adapter and use it in a SEPARATE Colab session
#     for GRPO (Runtime → Restart → reload SFT adapter → run GRPO).
#   - You do NOT intend to run Cell 10 in the same session.
# ──────────────────────────────────────────────────────────────────────────────

RUN_SFT = False   # ← set True only if doing SFT-only session (no GRPO after)

if not RUN_SFT:
    print('⏭  SFT skipped — proceeding directly to GRPO (recommended).')
    print('   Reason: SFT + GRPO in same session causes dtype conflict in matmul_lora.')
    print('   → Run Cell 8 next.')
else:
    import os
    os.makedirs(SFT_OUTPUT_DIR, exist_ok=True)
    from trl import SFTTrainer, SFTConfig

    FastLanguageModel.for_training(model)

    sft_trainer = SFTTrainer(
        model               = model,
        tokenizer           = tokenizer,
        train_dataset       = sft_dataset,
        dataset_text_field  = 'text',
        max_seq_length      = MAX_SEQ_LEN,
        dataset_num_proc    = 2,
        packing             = False,
        args = SFTConfig(
            per_device_train_batch_size  = SFT_BATCH,
            gradient_accumulation_steps  = SFT_GRAD_ACC,
            warmup_steps                 = SFT_WARMUP,
            num_train_epochs             = SFT_EPOCHS,
            learning_rate                = SFT_LR,
            fp16                         = not is_bfloat16_supported(),
            bf16                         = is_bfloat16_supported(),
            logging_steps                = 20,
            optim                        = 'adamw_8bit',
            weight_decay                 = 0.01,
            lr_scheduler_type            = 'linear',
            seed                         = 3407,
            output_dir                   = SFT_OUTPUT_DIR,
            save_strategy                = 'epoch',
            report_to                    = 'none',
        ),
    )

    print(f'SFT: {len(sft_dataset)} samples × {SFT_EPOCHS} epoch')
    sft_stats = sft_trainer.train()
    print(f'✓ SFT complete — loss: {sft_stats.metrics["train_loss"]:.4f}')

    # Save SFT adapter and download BEFORE running GRPO
    model.save_pretrained(SFT_OUTPUT_DIR)
    tokenizer.save_pretrained(SFT_OUTPUT_DIR)
    print(f'✓ SFT adapter saved to {SFT_OUTPUT_DIR}')
    print()
    print('⚠️  DO NOT run Cell 10 (GRPO) in this session.')
    print('   Download the SFT adapter, restart runtime, reload model, then do GRPO.')


In [ ]:
# CELL 8 — Prepare GRPO dataset
# Format: each record needs a 'prompt' (list of messages) + 'v_label' (for reward function)
# We pass all non-IDK GRPO records (2469 after filtering).
# Both Round 0 and Round 1 are included — different prompt structures.

def prepare_grpo_dataset(records):
    examples = []
    for r in records:
        if r.get('v_label') is None:
            continue  # skip IDK claims
        examples.append({
            'prompt': [
                {'role': 'system', 'content': r['system']},
                {'role': 'user',   'content': r['prompt']},
            ],
            'v_label':    float(r['v_label']),
            'agent_role': r['agent_role'],   # passed to reward fn as kwarg
            'brier_ref':  float(r.get('brier_reward') or 0.0),  # reference (not used in training)
        })
    return Dataset.from_list(examples)

grpo_dataset = prepare_grpo_dataset(grpo_data)

# Shuffle for training
grpo_dataset = grpo_dataset.shuffle(seed=3407)

print(f'GRPO dataset: {len(grpo_dataset)} records')
print(f'  v_label dist: '
      f'1.0={sum(1 for x in grpo_dataset if x["v_label"]==1.0)}, '
      f'0.5={sum(1 for x in grpo_dataset if x["v_label"]==0.5)}, '
      f'0.0={sum(1 for x in grpo_dataset if x["v_label"]==0.0)}')
print(f'  agent_role dist: '
      f'a={sum(1 for x in grpo_dataset if x["agent_role"]=="agent_a")}, '
      f'b={sum(1 for x in grpo_dataset if x["agent_role"]=="agent_b")}, '
      f'c={sum(1 for x in grpo_dataset if x["agent_role"]=="agent_c")}')
print(f'  ref avg brier: {sum(x["brier_ref"] for x in grpo_dataset)/len(grpo_dataset):.4f}')
print('\n✓ GRPO dataset ready')

In [ ]:
# CELL 9 — Reward functions
# Two reward signals:
#   1. brier_compliance_reward: main signal — calibrated accuracy vs judge label
#   2. format_reward: bonus for clean JSON structure (prevents reward hacking via malformed output)
#
# Both are passed to GRPOTrainer as a list.
# Final reward = weighted sum (weights set in GRPOConfig.reward_weights)

import json, re

VALID_VERDICTS = {'SUPPORTED', 'NOT_SUPPORTED', 'PARTIAL', 'IDK'}
REQUIRED_KEYS  = {'verdict', 'reasoning', 'evidence_cited', 'confidence_internal'}

def fix_json_newlines(s: str) -> str:
    """Escape literal newlines inside JSON string values (Qwen2.5 quirk at high temp)."""
    result, in_string, escape_next = [], False, False
    for ch in s:
        if escape_next:
            result.append(ch); escape_next = False
        elif ch == '\\':
            result.append(ch); escape_next = True
        elif ch == '"':
            result.append(ch); in_string = not in_string
        elif in_string and ch == '\n':
            result.append('\\n')
        elif in_string and ch == '\r':
            result.append('\\r')
        else:
            result.append(ch)
    return ''.join(result)

def try_parse_json(raw: str):
    """6-strategy parser — returns dict or None."""
    raw = raw.strip()
    for fn in [
        lambda r: json.loads(r),
        lambda r: json.loads(fix_json_newlines(r)),
        lambda r: json.loads(re.search(r'```(?:json)?\s*(\{.*?\})\s*```', r, re.DOTALL).group(1)),
        lambda r: json.loads(fix_json_newlines(re.search(r'```(?:json)?\s*(\{.*?\})\s*```', r, re.DOTALL).group(1))),
        lambda r: json.loads(re.search(r'\{.*\}', r, re.DOTALL).group(0)),
        lambda r: json.loads(fix_json_newlines(re.search(r'\{.*\}', r, re.DOTALL).group(0))),
    ]:
        try:
            result = fn(raw)
            if isinstance(result, dict):
                return result
        except Exception:
            continue
    return None

def p_for_brier(confidence: float, verdict: str) -> float:
    """Convert agent confidence_internal to P(claim is TRUE) for Brier scoring.
    confidence_internal = certainty in own verdict ≠ P(claim is true).
    NOT_SUPPORTED with conf=0.85 → P(true)=0.15 (agent is 85% sure it's NOT supported).
    PARTIAL always maps to 0.5 (partial support = 50% claim truth).
    """
    if verdict == 'NOT_SUPPORTED':
        return 1.0 - confidence
    elif verdict == 'PARTIAL':
        return 0.5
    elif verdict == 'SUPPORTED':
        return confidence
    else:  # IDK
        return 0.5


# ── Reward Function 1: Brier compliance reward ────────────────────────────────
def brier_compliance_reward(completions, v_label, **kwargs):
    """
    Main reward signal.
    Correct + confident call:   up to +1.0
    Uncertain call:             ~+0.5
    Wrong confident call:       down to -1.0
    Parse failure / no verdict: -1.0

    Args:
        completions: list of generated strings (one per sample in batch)
        v_label:     list of ground-truth labels (0.0 / 0.5 / 1.0)
    Returns:
        list of floats (rewards)
    """
    rewards = []
    for completion, vl in zip(completions, v_label):
        # Handle both string and message-dict completions
        if isinstance(completion, list):
            # GRPOTrainer passes as [{role, content}] — extract content
            completion = ' '.join(
                m.get('content', '') for m in completion
                if m.get('role') == 'assistant'
            )
        elif isinstance(completion, dict):
            completion = completion.get('content', '')

        parsed = try_parse_json(str(completion))

        if parsed is None:
            rewards.append(-1.0)  # complete parse failure
            continue

        verdict = parsed.get('verdict', '')
        if verdict not in VALID_VERDICTS:
            rewards.append(-0.5)  # invalid verdict label
            continue

        try:
            confidence = float(parsed.get('confidence_internal', 0.5))
            confidence = max(0.01, min(0.99, confidence))  # clamp to (0,1)
        except (TypeError, ValueError):
            confidence = 0.5

        p   = p_for_brier(confidence, verdict)
        r   = 1.0 - 2.0 * (p - float(vl)) ** 2   # symmetric Brier, range [-1, +1]
        rewards.append(float(round(r, 4)))

    return rewards


# ── Reward Function 2: Format reward ─────────────────────────────────────────
def format_reward(completions, **kwargs):
    """
    Bonus for producing clean, well-structured JSON.
    +0.1 if all required keys present in valid JSON.
    +0.0 if parseable but missing keys.
    -0.2 if completely fails to parse.

    Weighted low (0.1) — prevents gaming the format at expense of accuracy.
    """
    rewards = []
    for completion in completions:
        if isinstance(completion, list):
            completion = ' '.join(
                m.get('content', '') for m in completion
                if m.get('role') == 'assistant'
            )
        elif isinstance(completion, dict):
            completion = completion.get('content', '')

        parsed = try_parse_json(str(completion))
        if parsed is None:
            rewards.append(-0.2)
        elif REQUIRED_KEYS.issubset(set(parsed.keys())):
            rewards.append(0.1)   # all required keys present
        else:
            rewards.append(0.0)   # parseable but incomplete
    return rewards


# ── Quick sanity check ────────────────────────────────────────────────────────
test_cases = [
    ('{"verdict": "NOT_SUPPORTED", "reasoning": "no evidence", "evidence_cited": [], "confidence_internal": 0.85}', 0.0),
    ('{"verdict": "SUPPORTED", "reasoning": "clear evidence", "evidence_cited": [], "confidence_internal": 0.90}', 1.0),
    ('{"verdict": "PARTIAL", "reasoning": "partial", "evidence_cited": [], "confidence_internal": 0.70}', 0.5),
    ('not valid json at all', 0.0),
]

print('Reward function sanity check:')
print(f'{"Completion":<60} {"v_label":<8} {"brier_r":<10} {"fmt_r":<8}')
print('-' * 90)
for text, vl in test_cases:
    br = brier_compliance_reward([text], [vl])[0]
    fr = format_reward([text])[0]
    preview = text[:55] + '...' if len(text) > 55 else text
    print(f'{preview:<60} {vl:<8} {br:<10.4f} {fr:<8}')

print('\n✓ Reward functions ready')
print('  brier_compliance_reward: main signal, weight=1.0')
print('  format_reward:           structure bonus, weight=0.1')

In [ ]:
# CELL 10 — GRPO training
import torch, os, time
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import GRPOConfig, GRPOTrainer
os.makedirs(globals().get('GRPO_OUTPUT_DIR', '/content/grpo_output'), exist_ok=True)

# ── Apply Unsloth's GRPO patch ────────────────────────────────────────────────
try:
    from unsloth import PatchFastRL
    PatchFastRL("GRPO", FastLanguageModel)
    print('✓ PatchFastRL applied')
except (ImportError, AttributeError) as e:
    print(f'⚠ PatchFastRL not available: {e}')

# ── Detect model precision and match GRPOConfig to it ────────────────────────
# Unsloth enforces: GRPOConfig precision MUST match the model's loading precision.
# A100/H100 → bfloat16.  T4/V100 → float16.
# is_bfloat16_supported() returns True on A100, False on T4.
USE_BF16 = is_bfloat16_supported()
USE_FP16 = not USE_BF16
print(f'GPU precision: {"bfloat16 (A100)" if USE_BF16 else "float16 (T4)"}')
print(f'GRPOConfig:    bf16={USE_BF16}, fp16={USE_FP16}')
print()

# ── Defensive variable defaults ───────────────────────────────────────────────
GRPO_WARMUP_STEPS = globals().get('GRPO_WARMUP_STEPS', 50)
GRPO_LR           = globals().get('GRPO_LR',           5e-6)
GRPO_BATCH        = globals().get('GRPO_BATCH',        1)
GRPO_GRAD_ACC     = globals().get('GRPO_GRAD_ACC',     4)
GRPO_NUM_GEN      = globals().get('GRPO_NUM_GEN',      4)
GRPO_MAX_PROMPT   = globals().get('GRPO_MAX_PROMPT',   1024)
GRPO_MAX_COMP     = globals().get('GRPO_MAX_COMP',     512)
GRPO_MAX_GRAD     = globals().get('GRPO_MAX_GRAD',     0.1)
GRPO_EPOCHS       = globals().get('GRPO_EPOCHS',       2)
GRPO_OUTPUT_DIR   = globals().get('GRPO_OUTPUT_DIR',   '/content/grpo_output')

# ── Enable training mode ──────────────────────────────────────────────────────
FastLanguageModel.for_training(model)

# ── GRPOConfig ────────────────────────────────────────────────────────────────
grpo_config = GRPOConfig(
    # ── Optimizer ──────────────────────────────────────────────
    learning_rate               = GRPO_LR,
    adam_beta1                  = 0.9,
    adam_beta2                  = 0.99,
    weight_decay                = 0.1,
    warmup_steps                = GRPO_WARMUP_STEPS,
    lr_scheduler_type           = 'cosine',
    optim                       = 'adamw_8bit',
    max_grad_norm               = GRPO_MAX_GRAD,

    # ── Batch / steps ──────────────────────────────────────────
    per_device_train_batch_size = GRPO_BATCH,
    gradient_accumulation_steps = GRPO_GRAD_ACC,
    num_train_epochs            = GRPO_EPOCHS,

    # ── Generation ─────────────────────────────────────────────
    num_generations             = GRPO_NUM_GEN,
    max_prompt_length           = GRPO_MAX_PROMPT,
    max_completion_length       = GRPO_MAX_COMP,
    use_vllm                    = False,

    # ── Reward ─────────────────────────────────────────────────
    reward_weights              = [1.0, 0.1],

    # ── Logging / saving ───────────────────────────────────────
    logging_steps               = 10,
    save_steps                  = 100,
    output_dir                  = GRPO_OUTPUT_DIR,
    report_to                   = 'none',
    seed                        = 3407,

    # ── Precision: MUST match model loading precision ──────────
    # Unsloth checks this and raises TypeError if mismatched.
    # is_bfloat16_supported() = True on A100/H100, False on T4.
    fp16                        = USE_FP16,
    bf16                        = USE_BF16,
)

grpo_trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [brier_compliance_reward, format_reward],
    args             = grpo_config,
    train_dataset    = grpo_dataset,
)

total_steps = (len(grpo_dataset) // (GRPO_BATCH * GRPO_GRAD_ACC)) * GRPO_EPOCHS
gpu_name    = torch.cuda.get_device_properties(0).name
est_h       = total_steps * (2 if 'A100' in gpu_name else 5) / 3600
print(f'GRPO training config:')
print(f'  GPU:                 {gpu_name}')
print(f'  Dataset size:        {len(grpo_dataset)}')
print(f'  Num generations:     {GRPO_NUM_GEN} per prompt')
print(f'  Optimizer steps:     ~{total_steps} total')
print(f'  Effective batch:     {GRPO_BATCH * GRPO_GRAD_ACC}')
print(f'  Warmup steps:        {GRPO_WARMUP_STEPS}')
print(f'  Reward weights:      brier=1.0, format=0.1')
print(f'  Est. time:           ~{est_h:.1f}h')
print()
print('Starting GRPO training...')
print('Steps   1-50:  expect low/negative rewards (model learning JSON format)')
print('Steps  50-200: reward_mean rising toward 0.5+')
print('Steps 200+:    reward_mean rising toward 0.8+')
print()

t0 = time.time()
grpo_stats = grpo_trainer.train()
elapsed = time.time() - t0

print(f'\n✓ GRPO training complete')
print(f'  Total time: {elapsed/3600:.2f}h ({elapsed/60:.1f}min)')
if hasattr(grpo_stats, 'metrics'):
    for k, v in grpo_stats.metrics.items():
        if any(x in k for x in ['reward', 'loss', 'epoch']):
            print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
print(f'  GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB')


In [ ]:
# CELL 11 — Quick reward check on sample prompts
# Runs the trained model on 5 unseen prompts and computes live Brier reward.
# Verifies the model learned to output valid JSON with calibrated verdicts.
import random

FastLanguageModel.for_inference(model)  # enable inference optimizations

# Sample 5 records from GRPO dataset for quick eval
eval_indices = random.sample(range(len(grpo_dataset)), min(5, len(grpo_dataset)))
eval_records = [grpo_dataset[i] for i in eval_indices]

print('=== Quick Reward Check (trained model) ===')
print(f'{"#":<3} {"v_label":<8} {"agent":<8} {"verdict":<16} {"conf":<6} {"brier":<8} {"fmt":<6}')
print('-' * 65)

for idx, rec in enumerate(eval_records):
    # Format prompt
    messages = rec['prompt']  # already list of dicts
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors='pt').to('cuda')

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )

    parsed   = try_parse_json(generated)
    vl       = rec['v_label']
    agent    = rec['agent_role']

    if parsed:
        verdict = parsed.get('verdict', 'UNKNOWN')
        conf    = float(parsed.get('confidence_internal', 0.5))
        brier_r = brier_compliance_reward([generated], [vl])[0]
        fmt_r   = format_reward([generated])[0]
    else:
        verdict = 'PARSE_FAIL'
        conf    = 0.0
        brier_r = -1.0
        fmt_r   = -0.2

    print(f'{idx+1:<3} {vl:<8} {agent:<8} {verdict:<16} {conf:<6.2f} {brier_r:<8.4f} {fmt_r:<6}')

print()
print('✓ Check complete. If brier rewards are mostly > 0.7, GRPO worked.')
print('  If you see PARSE_FAIL, increase GRPO_EPOCHS or SFT_EPOCHS and retrain.')

In [ ]:
# CELL 12 — Save LoRA adapter + download
# Saves only the LoRA adapter weights (small — ~32MB for r=16)
# The base model stays frozen; the adapter is all you need for inference.
import os, shutil
from google.colab import files

os.makedirs(ADAPTER_DIR, exist_ok=True)

# Save LoRA adapter
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'✓ LoRA adapter saved to {ADAPTER_DIR}')

# Show adapter files
for f in sorted(os.listdir(ADAPTER_DIR)):
    sz = os.path.getsize(f'{ADAPTER_DIR}/{f}') / 1024
    print(f'  {f:<40} {sz:>8.1f} KB')

# Create zip for download
zip_path = '/content/mad_grpo_adapter.zip'
shutil.make_archive('/content/mad_grpo_adapter', 'zip', ADAPTER_DIR)
zip_size = os.path.getsize(zip_path) / (1024**2)
print(f'\n  Archive: {zip_path} ({zip_size:.1f} MB)')

print('\nDownloading adapter...')
files.download(zip_path)
print('✓ Done')

In [ ]:
# CELL 13 — (Optional) Push to Hugging Face Hub
# Run only if you want to share the adapter publicly.
# Requires a HF_TOKEN with write access.
# Skip if you just want to download locally (Cell 12 is enough).

PUSH_TO_HUB  = False        # set True to push
HF_USERNAME  = 'your-hf-username'
HF_REPO_NAME = 'mad-qwen25-7b-grpo-compliance'
HF_TOKEN     = ''           # paste your token here (write access required)

if PUSH_TO_HUB:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    
    repo_id = f'{HF_USERNAME}/{HF_REPO_NAME}'
    model.push_to_hub(repo_id, token=HF_TOKEN)
    tokenizer.push_to_hub(repo_id, token=HF_TOKEN)
    print(f'✓ Pushed to: https://huggingface.co/{repo_id}')
else:
    print('Skipped — set PUSH_TO_HUB=True to enable.')

## Training Summary

### What the reward function measures

```
Brier reward = 1 - 2 * (p_for_brier(confidence, verdict) - v_label)²

p_for_brier:
  SUPPORTED     → p = confidence          (high conf + SUPPORTED = high P(true))
  NOT_SUPPORTED → p = 1 - confidence      (high conf + NOT_SUPPORTED = low P(true))
  PARTIAL       → p = 0.5                 (always midpoint)
  IDK           → p = 0.5                 (no information)

Examples:
  NOT_SUPPORTED conf=0.85, v_label=0.0  →  p=0.15  →  R = 1-2*(0.15-0.0)² = +0.9550  ✓
  SUPPORTED     conf=0.90, v_label=1.0  →  p=0.90  →  R = 1-2*(0.90-1.0)² = +0.9800  ✓
  SUPPORTED     conf=0.90, v_label=0.0  →  p=0.90  →  R = 1-2*(0.90-0.0)² = -0.6200  ✗
  Parse failure                          →  R = -1.0                                    ✗
```

### Per-agent reward profiles (reference data)

| Agent | Role | Avg Brier (ref) | Strategy |
|-------|------|-----------------|----------|
| agent_a | Verifier | 0.877 | Argues FOR the claim — rewarded when claim IS supported |
| agent_b | Adversarial | 0.880 | Argues AGAINST — rewarded when claim is NOT supported |
| agent_c | Calibrator | 0.837 | Weighs both sides — rewarded for balanced PARTIAL verdicts |

### Data adequacy
- **GRPO: 2,484 rows** — above the 1,000+ threshold for LoRA GRPO. Round 1 prompts include peer debate context; Round 0 do not — both train structures help.
- **SFT: 1,221 rows** — sufficient for 1-epoch JSON format warm-up. All Brier > 0.5.
- **DPO: 77 pairs** — low count but excellent quality (avg margin 0.48). Use for optional post-GRPO alignment.

### Dtype bug fix log (for interview context)

| Bug | Cause | Fix |
|-----|-------|-----|
| `RuntimeError: Half and Float` (first) | `bnb-4bit` NF4 dequantizes to float32 during GRPO inference (no autocast). `matmul_lora` expected bfloat16 for `out` | Switch to full bfloat16 model (`load_in_4bit=False`) |
| `RuntimeError: Half and Float` (second) | Unsloth `'unsloth'` GC pre-allocates activation buffers in float16. During backward recompute, X arrives as float32 → `B.to(X.dtype)=float32` but `out=float16` | `use_gradient_checkpointing=False` (A100 has 40GB, GC not needed) |

### Interview story
```
Stage 1: Qwen2.5-14B-AWQ agents debate in 3-role MAD format (Verifier / Adversarial / Calibrator).
         Stochastic (temp 0.3–0.8) to ensure diverse debate signal.

Stage 2: Same model as deterministic judge (temp=0.0) labels each claim v_label ∈ {0.0, 0.5, 1.0}.
         Anti-bias: agent identity hidden, order randomized per claim, anti-majority instruction.

Stage 3: Qwen2.5-7B-Instruct fine-tuned with Unsloth GRPO (full bfloat16, no 4-bit).
         Reward = symmetric Brier score (corrected for NOT_SUPPORTED direction).
         GRPO optimizes calibrated accuracy across 414 claims × 3 agents × 2 rounds.
```